# Set Cover Benchmarking Comparison
This notebook compares three ansatz builders for the Set Cover problem:
1. **MomentumBuilder**: A growth-based ansatz construction using momentum gradients.
2. **MomentumMonteCarloBuilder**: Combines MomentumBuilder with Simulated Annealing.
3. **EfficientSU2**: A hardware-efficient ansatz (Qiskit standard).

The goal is to find the ground state (energy 0.0) of a 'Hard' Set Cover instance.

In [1]:
import numpy as np
import math
import time
import json
import heapq
from abc import ABC, abstractmethod
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import Parameter, ParameterVector
from qiskit.quantum_info import SparsePauliOp, partial_trace
from qiskit.primitives import StatevectorEstimator
from qiskit_aer.aerprovider import AerSimulator
from qiskit_algorithms import VQE
from qiskit_algorithms.optimizers import COBYLA
from qiskit.circuit.library import efficient_su2
from scipy.optimize import minimize
import matplotlib.pyplot as plt

## 1. Utilities
Core cost functions and gradient calculators.

In [2]:

def cost_func(params, circuit, hamiltonian, estimator):
    pub = (circuit, hamiltonian, params)
    cost = estimator.run([pub]).result()[0].data.evs
    return cost

def gradi(i, params, circuit, hamiltonian, estimator):
    delta = np.zeros(len(params))
    delta[i] = math.pi/2
    costp = cost_func(params+delta, circuit, hamiltonian, estimator)
    costm = cost_func(params-delta, circuit, hamiltonian, estimator)
    return (costp-costm)/2


## 2. Monte Carlo Optimization
Simulated Annealing implementation for MMC builder.

In [3]:

class MonteCarlo:
    @staticmethod
    def simulated_annealing(runs, params, ansatz, simulator, observables=None, estimator=None):
        def get_E(p):
            cost = cost_func(p, ansatz, observables, estimator)
            return cost[-1].item() if isinstance(cost, np.ndarray) else cost

        prev_E = get_E(params)
        best_params = params.copy()
        best_E = prev_E

        def T(t):
            c, a = 0.02, 0.01
            return c / (a + math.log(t))

        for t in range(1, runs):
            delta = np.random.normal(0, 0.1, len(params))
            params_new = params + delta
            E_new = get_E(params_new)
            delta_E = E_new - prev_E

            if delta_E <= 0:
                params = params_new
                prev_E = E_new
                if prev_E < best_E:
                    best_E = prev_E
                    best_params = params.copy()
            else:
                h = math.exp(-1 * delta_E / T(t))
                if np.random.random() < h:
                    params = params_new
                    prev_E = E_new
        return best_params


## 3. Set Cover Hamiltonian
Generates the Exactly Cover Hamiltonian.

In [4]:

def get_subset_Hamiltonian(universe, subsets):
    num_subsets = len(subsets)
    total_op = SparsePauliOp(["I" * num_subsets], coeffs=[0.0])

    for element in universe:
        relevant_indices = [i for i, s in enumerate(subsets) if element in s]
        pauli_id = "I" * num_subsets
        total_op += SparsePauliOp([pauli_id], coeffs=[1.0])
        
        for idx in relevant_indices:
            z_str = ["I"] * num_subsets
            z_str[num_subsets - 1 - idx] = "Z"
            total_op += SparsePauliOp(["".join(z_str)], coeffs=[0.5])
            total_op += SparsePauliOp([pauli_id], coeffs=[-0.5])
            
        relevant_indices.sort()
        for i_idx in range(len(relevant_indices)):
            for j_idx in range(i_idx + 1, len(relevant_indices)):
                u, v = relevant_indices[i_idx], relevant_indices[j_idx]
                total_op += SparsePauliOp([pauli_id], coeffs=[0.5])
                z_u_str = ["I"] * num_subsets
                z_u_str[num_subsets - 1 - u] = "Z"
                total_op += SparsePauliOp(["".join(z_u_str)], coeffs=[-0.5])
                z_v_str = ["I"] * num_subsets
                z_v_str[num_subsets - 1 - v] = "Z"
                total_op += SparsePauliOp(["".join(z_v_str)], coeffs=[-0.5])
                z_uv_str = ["I"] * num_subsets
                z_uv_str[num_subsets - 1 - u] = "Z"
                z_uv_str[num_subsets - 1 - v] = "Z"
                total_op += SparsePauliOp(["".join(z_uv_str)], coeffs=[0.5])

    return total_op.simplify()


## 4. Momentum Growth Logic
Core growth mechanism for Momentum builders.

In [5]:

def momen_layer(it, n, momentum, radius=1, keep=2):
    lay = QuantumCircuit(n)
    params, inds = [], []
    for i in range(keep):
        angle = Parameter(f"it-{it}, {i}")
        ind = momentum[i][1]
        params.append(1)
        inds.append(ind)
        lay.rx(angle, ind)
        for r in range(1, radius+1):
            if ind + r < n: lay.cx(ind, ind+r)
            if ind - r >= 0: lay.cx(ind, ind-r)
    return lay, params, inds

def momentum_builder_logic(params, inds, ansatz, circuit, observables, estimator, beta1, beta2, iters=2):
    n = circuit.num_qubits
    M = np.zeros(len(params))
    currCirc = circuit.compose(ansatz)
    for it in range(iters):
        accumulator = []
        for i in range(len(params)):
            grad_i = abs(gradi(i, params, currCirc, observables, estimator)[-1]).item()
            M[i] = beta1 * M[i] + (1 - beta1) * grad_i
            heapq.heappush(accumulator, (M[i], inds[i]))
        mLayer, nparams, ninds = momen_layer(it, n, accumulator)
        params += nparams
        inds += ninds
        M = np.concatenate((M, [0]*len(nparams)))
        ansatz = ansatz.compose(mLayer)
        currCirc = circuit.compose(ansatz)
    return circuit.compose(ansatz)


## 5. Builders
Unified interface for benchmarks.

In [6]:

class AnsatzBuilder(ABC):
    def __init__(self, hamiltonian):
        self.hamiltonian = hamiltonian
        self.circuit = None
        self.initial_point = None

    @abstractmethod
    def build(self): pass

    def getCircuit(self):
        if self.circuit is None: self.circuit = self.build()
        return self.circuit

class MomentumBuilder(AnsatzBuilder):
    def build(self):
        n = self.hamiltonian.num_qubits
        ansatz, circuit = QuantumCircuit(n), QuantumCircuit(n)
        for i in range(n): ansatz.rx(Parameter(f"a{i}"), i)
        obs = [*self.hamiltonian.paulis, self.hamiltonian]
        self.circuit = momentum_builder_logic([1]*n, list(range(n)), ansatz, circuit, obs, StatevectorEstimator(), 0.9, 0.99)
        return self.circuit

class MomentumMonteCarloBuilder(AnsatzBuilder):
    def build(self):
        n = self.hamiltonian.num_qubits
        ansatz, circuit = QuantumCircuit(n), QuantumCircuit(n)
        for i in range(n): ansatz.rx(Parameter(f"a{i}"), i)
        obs = [*self.hamiltonian.paulis, self.hamiltonian]
        
        # Build
        mid_circuit = momentum_builder_logic([1]*n, list(range(n)), ansatz, circuit, obs, StatevectorEstimator(), 0.9, 0.99)
        
        # SA Optimization
        sim = AerSimulator(method='statevector')
        opt_params = MonteCarlo.simulated_annealing(100, np.ones(len(mid_circuit.parameters)), mid_circuit, sim, obs, StatevectorEstimator())
        
        self.circuit = mid_circuit
        self.initial_point = opt_params
        return self.circuit

class EfficientSU2(AnsatzBuilder):
    def build(self):
        self.circuit = efficient_su2(self.hamiltonian.num_qubits, reps=2)
        return self.circuit


## 6. Evaluator
VQE based evaluation framework.

In [7]:

def evaluateBuilder(builder_class, problems):
    results = []
    est = StatevectorEstimator()
    opt = COBYLA(maxiter=500)
    
    for i, (h, exact) in enumerate(problems):
        hamiltonian = -h # For minimization
        builder = builder_class(hamiltonian)
        start = time.time()
        circuit = builder.getCircuit()
        b_time = time.time() - start
        
        vqe = VQE(estimator=est, ansatz=circuit, optimizer=opt, initial_point=builder.initial_point)
        vs = time.time()
        vqe_res = vqe.compute_minimum_eigenvalue(operator=hamiltonian)
        vt = time.time() - vs
        
        v_energy = -1 * vqe_res.eigenvalue.real
        results.append({
            "builder": builder_class.__name__,
            "total_time": b_time + vt,
            "vqe_energy": v_energy,
            "exact_energy": exact
        })
    return results


## 7. Run Benchmarks
Comparing the builders on the 'Hard' 8-qubit Set Cover instance.

In [8]:

# Define hard problem
universe = [1, 2, 3, 4, 5, 6]
subsets = [{1, 2}, {3, 4}, {5, 6}, {1, 3, 5}, {2, 4, 6}, {1, 4}, {2, 5}, {3, 6}]
H = get_subset_Hamiltonian(universe, subsets)
problems = [(-H, 0.0)]

builders = [MomentumBuilder, MomentumMonteCarloBuilder, EfficientSU2]
num_trials = 10
all_results = []

for b in builders:
    # Handle display names for plotting
    display_name = "Efficient SU2" if b.__name__ == "EfficientSU2" else b.__name__
    print(f"Evaluating {display_name}...")
    for t in range(num_trials):
        print(f"  Trial {t+1}/{num_trials}")
        res = evaluateBuilder(b, problems)
        for r in res: 
            r['trial'] = t+1
            r['display_name'] = display_name
        all_results.extend(res)

# Process Results
summary_data = []
for display_name in ["MomentumBuilder", "MomentumMonteCarloBuilder", "Efficient SU2"]:
    subset = [r for r in all_results if r['display_name'] == display_name]
    final_energies = [-r['vqe_energy'] for r in subset]
    times = [r['total_time'] for r in subset]
    
    summary_data.append({
        'Method': display_name,
        'Final Energy': np.mean(final_energies),
        'Mean Time (s)': np.mean(times)
    })

import pandas as pd
df_summary = pd.DataFrame(summary_data)
# Formatted to 4 decimal points
pd.options.display.float_format = '{:,.4f}'.format
display(df_summary)


Evaluating MomentumBuilder...
  Trial 1/10
  Trial 2/10
  Trial 3/10
  Trial 4/10
  Trial 5/10
  Trial 6/10
  Trial 7/10
  Trial 8/10
  Trial 9/10
  Trial 10/10
Evaluating MomentumMonteCarloBuilder...
  Trial 1/10
  Trial 2/10
  Trial 3/10
  Trial 4/10
  Trial 5/10
  Trial 6/10
  Trial 7/10
  Trial 8/10
  Trial 9/10
  Trial 10/10
Evaluating Efficient SU2...
  Trial 1/10
  Trial 2/10
  Trial 3/10
  Trial 4/10
  Trial 5/10
  Trial 6/10
  Trial 7/10
  Trial 8/10
  Trial 9/10
  Trial 10/10


,Method,Final Energy,Mean Time (s)
0,MomentumBuilder,0.4000,0.7222
1,MomentumMonteCarloBuilder,0.2000,1.6748
2,Efficient SU2,0.4317,1.1150


## 8. Visualization
Separate comparisons for Final Energy and Mean Execution Time.

In [9]:

import plotly.graph_objects as go

# 1. Final Energy Plot
fig_energy = go.Figure()
fig_energy.add_trace(go.Bar(
    x=df_summary['Method'],
    y=df_summary['Final Energy'],
    name='Final Energy',
    marker_color='indianred',
    text=[f"{v:.4f}" for v in df_summary['Final Energy']],
    textposition='auto',
))
fig_energy.update_layout(
    title='Comparison: Final Energy (Lower is Better)',
    yaxis_title='Final Energy',
    template='plotly_dark',
    height=400
)
fig_energy.show()

# 2. Mean Time Plot
fig_time = go.Figure()
fig_time.add_trace(go.Bar(
    x=df_summary['Method'],
    y=df_summary['Mean Time (s)'],
    name='Mean Time',
    marker_color='royalblue',
    text=[f"{v:.3f}s" for v in df_summary['Mean Time (s)']],
    textposition='auto',
))
fig_time.update_layout(
    title='Comparison: Mean Execution Time',
    yaxis_title='Time (Seconds)',
    template='plotly_dark',
    height=400
)
fig_time.show()
